<a href="https://colab.research.google.com/github/prince-musonda/my-journey-to-artificial-intelligence/blob/main/stegno_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
marcozuppelli_stegoimagesdataset_path = kagglehub.dataset_download('marcozuppelli/stegoimagesdataset')

print('Data source import complete.')

100%|██████████| 1.51G/1.51G [01:12<00:00, 22.2MB/s]

Extracting files...


Data source import complete.


# import necessary libraries

In [ ]:
from tqdm import tqdm
import torch
from torch import nn
import torchvision
from torchvision import models
try:
  import torchinfo
except:
  !pip install torchinfo
  import torchinfo

In [ ]:
weights = models.EfficientNet_B5_Weights.DEFAULT
img_tranforms = weights.transforms()
model = models.efficientnet_b5(weights=weights)


Downloading: "https://download.pytorch.org/models/efficientnet_b5_lukemelas-1a07897c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b5_lukemelas-1a07897c.pth
100%|██████████| 117M/117M [00:00<00:00, 189MB/s]


In [ ]:
torchinfo.summary(model=model,
                  input_size=(32,3,244,244),
                  col_names=["input_size", "output_size", "trainable"],
                 row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape               Output Shape              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 244, 244]         [32, 1000]                True
├─Sequential (features)                                      [32, 3, 244, 244]         [32, 2048, 8, 8]          True
│    └─Conv2dNormActivation (0)                              [32, 3, 244, 244]         [32, 48, 122, 122]        True
│    │    └─Conv2d (0)                                       [32, 3, 244, 244]         [32, 48, 122, 122]        True
│    │    └─BatchNorm2d (1)                                  [32, 48, 122, 122]        [32, 48, 122, 122]        True
│    │    └─SiLU (2)                                         [32, 48, 122, 122]        [32, 48, 122, 122]        --
│    └─Sequential (1)                                        [32, 48, 122, 122]        [32, 24, 122, 122]        True
│    │    └─MBConv (0)                               

# **Freeze feature layer**

In [ ]:
for param in model.features.parameters():
    param.requires_grad = False

# replace classification layer

In [ ]:
model.classifier = nn.Sequential(nn.Dropout(p=0.4, inplace=True),
                                nn.Linear(in_features=2048, out_features=2, bias=True))

# set up loss function and optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# **create dataloaders**

In [ ]:
import os

def create_dataloader(training_data_path, testing_data_path, img_transforms, batch_size=32):

    """
    training_path: the file path to your data's training folder
    test_path: the file to your data's testing folder
    transforms: image transforms to be applied on your image data
    batch: batch size

    return:   train_dataLoader, test_dataLoader, classes
    """

    train_dataset = torchvision.datasets.ImageFolder(training_data_path, transform=img_transforms)
    test_dataset = torchvision.datasets.ImageFolder(testing_data_path, transform=img_transforms)

    train_data_loader = torch.utils.data.DataLoader(train_dataset,
                                                    batch_size=batch_size,
                                                    shuffle=True,
                                                    num_workers=os.cpu_count())

    test_data_loader = torch.utils.data.DataLoader(test_dataset,
                                                  batch_size = batch_size,
                                                  shuffle=False,
                                                  num_workers=os.cpu_count())

    classes = train_dataset.classes
    test_classes = test_dataset.classes

    return train_data_loader, test_data_loader, classes

In [ ]:
# remove some unnecessary folders in the test data folder
!rm -r /root/.cache/kagglehub/datasets/marcozuppelli/stegoimagesdataset/versions/2/test/test/stego_zip/
!rm -r /root/.cache/kagglehub/datasets/marcozuppelli/stegoimagesdataset/versions/2/test/test/stego_b64


In [ ]:
training_data_path = marcozuppelli_stegoimagesdataset_path + "/train/train"
testing_data_path = marcozuppelli_stegoimagesdataset_path + "/test/test"

train_data_loader, test_data_loader, classes = create_dataloader(training_data_path,
                                                                 testing_data_path,
                                                                 batch_size=10,
                                                                 img_transforms= img_tranforms)
classes



['clean', 'stego']

# set up **training step** and **test step**

In [ ]:
def accuracy_fn(true, pred):
  correct = torch.eq(true,pred).sum().item()
  accuracy = (correct/len(true)) * 100
  return accuracy

def train_step( model: nn.Module,
                train_data_loader: torch.utils.data.DataLoader,
                optimizer:torch.optim.Optimizer,
                loss_fn: nn.Module,
                device:torch.device):

  train_loss, train_accuracy = 0, 0
  model.to(device)
  # put model in taining mode
  model.train()
  for batch, (X,y) in enumerate(train_data_loader):
    # send data to target device
    X, y = X.to(device), y.to(device)
    # forward propagation
    prediction = model(X)
    # calculate loss
    loss = loss_fn(prediction, y)
    train_loss += loss.item()
    # calculate accuracy
    train_pred = prediction.softmax(dim=1).argmax(dim=1)
    train_accuracy+= accuracy_fn(y,train_pred)
    # back probagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # show the number of batches seen
    if (batch+1) % 50 == 0:
      print(f"seen {(batch+1) * 10} / {len(train_data_loader) * 10} samples")

  # calculate average loss an accuracy per batch
  train_loss /= len(train_data_loader)
  train_accuracy /= len(train_data_loader)

  return train_loss, train_accuracy





In [ ]:
def test_step( model: nn.Module,
                test_data_loader: torch.utils.data.DataLoader,
                loss_fn: nn.Module,
                device:torch.device):

  test_loss, test_accuracy = 0, 0
  model.to(device)
  # put model in evaluation
  model.eval()
  with torch.inference_mode():

    for  (X,y) in test_data_loader:
      # send data to target device
      X, y = X.to(device), y.to(device)
      # forward propagation
      prediction = model(X)
      # calculate loss
      loss = loss_fn(prediction, y)
      test_loss += loss.item()
      # calculate accuracy
      test_pred = prediction.softmax(dim=1).argmax(dim=1)
      test_accuracy+= accuracy_fn(y,test_pred)

    # calculate average loss an accuracy per batch
    test_loss /= len(test_data_loader)
    test_accuracy /= len(test_data_loader)

    return test_loss, test_accuracy

# training time

In [ ]:
# mount google drive so that we can be saving the modelslearning after each epoch
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
epoches = 3
torch.manual_seed(42)
torch.cuda.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

training_losses = []
training_accuracies = []
testing_losses = []
testing_accuracies = []

for epoch in tqdm(range(epoches)):
  # train_model
  train_loss, train_accuracy = train_step(model,train_data_loader,optimizer,loss_fn,device)
  training_accuracies.append(train_accuracy)
  training_losses.append(train_loss)
  # save model learnt weights after each epoch
  torch.save(model.state_dict(),"/content/drive/MyDrive/Colab Notebooks/saved_models/stegno_model.pt")
  print(f"train loss {train_loss:.5f}  accuracy: {train_accuracy:.2f}%")
  # evaluate model
  test_loss, test_accuracy =  test_step(model,test_data_loader,loss_fn,device)
  testing_losses.append(test_loss)
  testing_accuracies.append(test_accuracy)
  print(f"test loss {test_loss:.5f}  accuracy: {test_accuracy:.2f}% \n\n")

  0%|          | 0/3 [00:00<?, ?it/s]

seen 500 / 16000 samples
seen 1000 / 16000 samples
seen 1500 / 16000 samples
seen 2000 / 16000 samples
seen 2500 / 16000 samples
seen 3000 / 16000 samples
seen 3500 / 16000 samples
seen 4000 / 16000 samples
seen 4500 / 16000 samples
seen 5000 / 16000 samples
seen 5500 / 16000 samples
seen 6000 / 16000 samples
seen 6500 / 16000 samples
seen 7000 / 16000 samples
seen 7500 / 16000 samples
seen 8000 / 16000 samples
seen 8500 / 16000 samples
seen 9000 / 16000 samples
seen 9500 / 16000 samples
seen 10000 / 16000 samples
seen 10500 / 16000 samples
seen 11000 / 16000 samples
seen 11500 / 16000 samples
seen 12000 / 16000 samples
seen 12500 / 16000 samples
seen 13000 / 16000 samples
seen 13500 / 16000 samples
seen 14000 / 16000 samples
seen 14500 / 16000 samples
seen 15000 / 16000 samples
seen 15500 / 16000 samples
seen 16000 / 16000 samples
train loss 0.53281  accuracy: 73.71%


 33%|███▎      | 1/3 [09:36<19:12, 576.02s/it]

test loss 0.51552  accuracy: 74.01% 


seen 500 / 16000 samples
seen 1000 / 16000 samples
seen 1500 / 16000 samples
seen 2000 / 16000 samples
seen 2500 / 16000 samples
seen 3000 / 16000 samples
seen 3500 / 16000 samples
seen 4000 / 16000 samples
seen 4500 / 16000 samples
seen 5000 / 16000 samples
seen 5500 / 16000 samples
seen 6000 / 16000 samples
seen 6500 / 16000 samples
seen 7000 / 16000 samples
seen 7500 / 16000 samples
seen 8000 / 16000 samples
seen 8500 / 16000 samples
seen 9000 / 16000 samples
seen 9500 / 16000 samples
seen 10000 / 16000 samples
seen 10500 / 16000 samples
seen 11000 / 16000 samples
seen 11500 / 16000 samples
seen 12000 / 16000 samples
seen 12500 / 16000 samples
seen 13000 / 16000 samples
seen 13500 / 16000 samples
seen 14000 / 16000 samples
seen 14500 / 16000 samples
seen 15000 / 16000 samples
seen 15500 / 16000 samples
seen 16000 / 16000 samples
train loss 0.53656  accuracy: 73.24%


 67%|██████▋   | 2/3 [19:11<09:35, 575.78s/it]

test loss 0.51810  accuracy: 74.24% 


seen 500 / 16000 samples
seen 1000 / 16000 samples
seen 1500 / 16000 samples
seen 2000 / 16000 samples
seen 2500 / 16000 samples
seen 3000 / 16000 samples
seen 3500 / 16000 samples
seen 4000 / 16000 samples
seen 4500 / 16000 samples
seen 5000 / 16000 samples
seen 5500 / 16000 samples
seen 6000 / 16000 samples
seen 6500 / 16000 samples
seen 7000 / 16000 samples
seen 7500 / 16000 samples
seen 8000 / 16000 samples
seen 8500 / 16000 samples
seen 9000 / 16000 samples
seen 9500 / 16000 samples
seen 10000 / 16000 samples
seen 10500 / 16000 samples
seen 11000 / 16000 samples
seen 11500 / 16000 samples
seen 12000 / 16000 samples
seen 12500 / 16000 samples
seen 13000 / 16000 samples
seen 13500 / 16000 samples
seen 14000 / 16000 samples
seen 14500 / 16000 samples
seen 15000 / 16000 samples
seen 15500 / 16000 samples
seen 16000 / 16000 samples
train loss 0.55062  accuracy: 72.58%


100%|██████████| 3/3 [28:43<00:00, 574.60s/it]

test loss 0.52152  accuracy: 73.95% 




# **model 2**

In [ ]:
import copy

model_2 = copy.deepcopy(model)
model_2.classifier

Sequential(
  (0): Dropout(p=0.4, inplace=True)
  (1): Linear(in_features=2048, out_features=2, bias=True)
)

In [ ]:
# enable grad in model 2
for param in model_2.features.parameters():
  param.requires_grad = False


# chnage classifer part
model_2.classifer = nn.Sequential(
    nn.Dropout(p=0.2,inplace=True),
    nn.Linear(in_features=2048, out_features=1000,bias=True),
    nn.ReLU(),
    nn.Linear(in_features=1000, out_features=2, bias=True),
    nn.ReLU()
)


In [ ]:
# set up optimizer
optimizer_2 = torch.optim.Adam(model_2.parameters(), lr=0.001)

In [ ]:
# training loop
training_losses_2 = []
training_accuracies_2 = []
testing_losses_2 = []
testing_accuracies_2 = []
epochs = 3
for epoch in tqdm(range(epochs)):
  train_loss, train_accuracy=train_step(model_2,train_data_loader,optimizer_2, loss_fn,device)
  training_accuracies_2.append(train_accuracy)
  training_losses_2.append(train_loss)
  # save model learnt weights after each epoch
  torch.save(model.state_dict(),"/content/drive/MyDrive/Colab Notebooks/saved_models/stegno_model2.pt")
  print(f"train loss {train_loss:.5f}  accuracy: {train_accuracy:.2f}%")
  # evaluate model
  test_loss, test_accuracy =  test_step(model,test_data_loader,loss_fn,device)
  testing_losses_2.append(test_loss)
  testing_accuracies_2.append(test_accuracy)
  print(f"test loss {test_loss:.5f}  accuracy: {test_accuracy:.2f}% \n\n")

  0%|          | 0/3 [00:00<?, ?it/s]

seen 500 / 16000 samples
seen 1000 / 16000 samples
seen 1500 / 16000 samples
seen 2000 / 16000 samples
seen 2500 / 16000 samples
seen 3000 / 16000 samples
seen 3500 / 16000 samples
seen 4000 / 16000 samples
seen 4500 / 16000 samples
seen 5000 / 16000 samples
seen 5500 / 16000 samples
seen 6000 / 16000 samples
seen 6500 / 16000 samples
seen 7000 / 16000 samples
seen 7500 / 16000 samples
seen 8000 / 16000 samples
seen 8500 / 16000 samples
seen 9000 / 16000 samples
seen 9500 / 16000 samples
seen 10000 / 16000 samples
seen 10500 / 16000 samples
seen 11000 / 16000 samples
seen 11500 / 16000 samples
seen 12000 / 16000 samples
seen 12500 / 16000 samples
seen 13000 / 16000 samples
seen 13500 / 16000 samples
seen 14000 / 16000 samples
seen 14500 / 16000 samples
seen 15000 / 16000 samples
seen 15500 / 16000 samples
seen 16000 / 16000 samples
train loss 0.55469  accuracy: 72.14%


 33%|███▎      | 1/3 [09:34<19:09, 574.91s/it]

test loss 0.52143  accuracy: 72.76% 


seen 500 / 16000 samples
seen 1000 / 16000 samples
seen 1500 / 16000 samples
seen 2000 / 16000 samples
seen 2500 / 16000 samples


 33%|███▎      | 1/3 [10:41<21:23, 641.97s/it]


KeyboardInterrupt: 